## STATUS BANNER (added 2026-07-25)

**This notebook has NOT been executed with this code.** All outputs below are empty --
this is the current, fixed source, staged in `run-final/` for the next Colab run, not
a record of a completed run.

**What changed here today:** Replaces the previous executed copy. No code changes from today's fixes apply to this notebook -- re-run anyway so the whole batch is regenerated together as one coherent, reproducible set.

**Run order for the full batch** (see `docs/module4_remediation_plan.md`):
1. `mdc_preprocess_vNext_output.ipynb`
2. `mdc_preprocess_vNext_mc_output.ipynb`
3. `mdc_model_vNext_output.ipynb`
4. `mdc_drift_aware_output.ipynb`
5. `mdc_baselines_output.ipynb` (run last -- its comparison table needs steps 3 and 4's outputs)

**Previous executed run (real numbers, kept for reference/reconciliation):**
`notebook/run-final-legacy-2026-07-24/` -- default ROC 0.7042, HPO 0.8138, multi-seed
0.7529+/-0.023, stream ordering already `timestamp`, `claim_ok=False`, fine-tune AUC
delta reported as **NaN** (real last-30% chronological slice was 100% attack, zero
benign windows -- see that notebook's own cell 16 output). That NaN finding is a
result in its own right and should be written up, not silently dropped once you
have a fresh run.


# MDC Preprocessing vNext-MC — multiclass labels + window timestamps

**Self-contained:** upload **only this notebook**. Drive I/O helpers are a readable code cell; label map is inline JSON (no base64 / extra `.py`).

**Drive layout:** `My Drive / vNEXT_test / processed / windows_vnext_mc.npz`

Artifacts stage under `/content/vNEXT_test_local/processed/` then verified-push to Drive.


## 0. Setup & config

In [ ]:
import sys, os, subprocess, gc, json
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
import warnings
warnings.filterwarnings('ignore')

try:
    import google.colab
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

DRIVE_PROJECT_FOLDER = 'vNEXT_test'

if _IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'kagglehub[pandas-datasets]', 'joblib'], check=False)
    # Stage locally first (FUSE-safe); push cell copies to Drive
    OUTPUT_DIR = '/content/vNEXT_test_local/processed'
else:
    OUTPUT_DIR = os.path.normpath(f'../outputs/{DRIVE_PROJECT_FOLDER}/processed')
os.makedirs(OUTPUT_DIR, exist_ok=True)

VERSION       = 'vnext'
RANDOM_STATE  = 42
BENIGN_LABEL  = 0

TRAIN_RATIO   = 0.70
VAL_RATIO     = 0.15
TEST_RATIO    = 0.15

MIN_FLOWS          = 10_000
GAP_THRESHOLD      = 60
VARIANCE_THRESHOLD = 0.01
CORR_THRESHOLD     = 0.95

BUCKET_FREQ           = '15s'
BUCKET_AGG            = 'mean_max_std'
WINDOW_SIZE           = 10
STRIDE                = 2
ATTACK_FRAC_THRESHOLD = 0.5
MAX_GAP_BUCKETS       = 4
POST_SCALE_CLIP       = 10.0

DROP_COLS = [
    'Flow ID', 'Dst IP', 'Timestamp',
    'Fwd URG Flags', 'Bwd URG Flags',
    'URG Flag Count', 'CWR Flag Count', 'ECE Flag Count',
]

print(f'preprocess_{VERSION}_mc config')
print(f'  IN_COLAB         = {_IN_COLAB}')
print(f'  PROJECT          = {DRIVE_PROJECT_FOLDER}')
print(f'  OUTPUT_DIR       = {OUTPUT_DIR}  (local staging)')
print(f'  Drive UI target  = My Drive > {DRIVE_PROJECT_FOLDER} > processed')
print(f'  bucket_agg       = {BUCKET_AGG}')
print(f'  window           = {WINDOW_SIZE} x {BUCKET_FREQ} = {WINDOW_SIZE*15}s')
print(f'  stride           = {STRIDE} x {BUCKET_FREQ} = {STRIDE*15}s')
print(f'  attack_frac_thr  = {ATTACK_FRAC_THRESHOLD}')
print(f'  max_gap_buckets  = {MAX_GAP_BUCKETS}')
print(f'  post_scale_clip  = +/-{POST_SCALE_CLIP}')
print(f'  MC labels        : y_val_multiclass + y_test_multiclass + ts_*')


## 1. Load raw data (Kaggle or local CSV)

In [ ]:
from pathlib import Path

KAGGLE_DATASET = 'yigitsever/misuse-detection-in-containers-dataset'
KAGGLE_CSV     = 'MDC dataset.csv'

if _IN_COLAB:
    try:
        from google.colab import userdata
        for k in ('KAGGLE_USERNAME', 'KAGGLE_KEY'):
            v = userdata.get(k)
            if v: os.environ[k] = v
    except Exception:
        pass

import kagglehub
from kagglehub import KaggleDatasetAdapter
try:
    df = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, KAGGLE_DATASET, KAGGLE_CSV)
except Exception:
    root = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    candidates = sorted(root.rglob('*.csv'), key=lambda p: p.stat().st_size, reverse=True)
    df = pd.read_csv(candidates[0])

df.columns = df.columns.str.strip()
_labels = sorted(df['Label'].unique())
print(f'Raw shape: {df.shape}   labels={_labels}')

## 2. Container filter + timestamp + session assignment

In [ ]:
counts   = df['Src IP'].value_counts()
keep_ips = counts[counts >= MIN_FLOWS].index.tolist()
df       = df[df['Src IP'].isin(keep_ips)].copy().reset_index(drop=True)
print(f'After container filter: {len(df):,} rows  /  {len(keep_ips)} containers')

df['ts'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df = df.dropna(subset=['ts']).sort_values(['Src IP', 'ts']).reset_index(drop=True)

def assign_sessions(g, threshold=60):
    gap = g['ts'].diff().dt.total_seconds().fillna(0)
    sn  = (gap > threshold).cumsum()
    g['session_id'] = g['Src IP'].astype(str) + '_s' + sn.astype(str)
    return g

df = df.groupby('Src IP', group_keys=False).apply(assign_sessions, threshold=GAP_THRESHOLD)
df['time_gap_s'] = df.groupby('Src IP')['ts'].diff().dt.total_seconds()
_n_sess = df['session_id'].nunique()
print(f'Sessions: {_n_sess:,}')
print(df.groupby('Src IP')['session_id'].nunique().to_string())

## 3. Session-level random split + benign-only fit setup

Random per-container split (TRAIN_RATIO=0.70) → train ∪ holdout. All subsequent statistics
(median, variance, correlation, clip, scaler) are fit on **training benign flows only**.

In [ ]:
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)
gc.collect()

_ss = df[['Src IP', 'session_id', 'ts']].drop_duplicates('session_id').copy()
rng = np.random.default_rng(RANDOM_STATE)
train_sessions, holdout_sessions = set(), set()
for src_ip, grp in _ss.groupby('Src IP'):
    sess = grp['session_id'].tolist()
    n_train = max(1, int(len(sess) * TRAIN_RATIO))
    shuffled = rng.permutation(sess)
    train_sessions.update(shuffled[:n_train])
    holdout_sessions.update(shuffled[n_train:])
del _ss; gc.collect()

train_mask  = df['session_id'].isin(train_sessions)
meta_cols   = ['Src IP', 'session_id', 'Label', 'ts']
meta_train  = df.loc[train_mask, meta_cols].reset_index(drop=True)
meta_holdout= df.loc[~train_mask, meta_cols].reset_index(drop=True)

non_meta = [c for c in df.columns if c not in meta_cols + ['time_gap_s']]
df_train = df.loc[train_mask, non_meta].copy().reset_index(drop=True)
df_hold  = df.loc[~train_mask, non_meta].copy().reset_index(drop=True)
del df; gc.collect()

benign_idx = meta_train['Label'].eq(BENIGN_LABEL).values
_hold_atk_pct = (meta_holdout['Label'] != 0).mean()*100
print(f'train flows: {len(df_train):,}   benign: {benign_idx.sum():,}  ({benign_idx.mean()*100:.1f}%)')
print(f'hold  flows: {len(df_hold):,}    holdout attack rate: {_hold_atk_pct:.1f}%')

## 4. Inf/NaN/dtype hygiene

In [ ]:
def to_numeric_safe(df_):
    for c in df_.columns:
        df_[c] = pd.to_numeric(df_[c], errors='coerce').astype(np.float32)
    return df_

df_train = to_numeric_safe(df_train)
df_hold  = to_numeric_safe(df_hold)

df_train = df_train.replace([np.inf, -np.inf], np.nan)
df_hold  = df_hold .replace([np.inf, -np.inf], np.nan)

train_medians = df_train[benign_idx].median(numeric_only=True).fillna(0.0).astype(np.float32)
df_train = df_train.fillna(train_medians)
df_hold  = df_hold .fillna(train_medians)
print(f'Filled NaNs with benign-train medians ({len(train_medians)} columns)')

## 5. Variance threshold filter (fit on benign train)

In [ ]:
feat_names_before_var = df_train.columns.tolist()
vt = VarianceThreshold(threshold=VARIANCE_THRESHOLD)
vt.fit(df_train.loc[benign_idx].to_numpy(dtype=np.float32))
var_mask = vt.get_support()
keep_var = [c for c, ok in zip(feat_names_before_var, var_mask) if ok]
df_train = df_train[keep_var]; df_hold = df_hold[keep_var]
print(f'VarianceThreshold {VARIANCE_THRESHOLD} -> kept {df_train.shape[1]} / {len(feat_names_before_var)}')

## 6. Correlation filter (benign-train sample)

In [ ]:
n_sample = min(100_000, int(benign_idx.sum()))
sample   = df_train.loc[benign_idx].sample(n=n_sample, random_state=RANDOM_STATE).astype(np.float32)
corr     = sample.corr(numeric_only=True).abs()
upper    = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
drop_corr= [c for c in upper.columns if any(upper[c] > CORR_THRESHOLD)]
df_train = df_train.drop(columns=drop_corr, errors='ignore')
df_hold  = df_hold .drop(columns=drop_corr, errors='ignore')
del sample, corr, upper; gc.collect()
print(f'Correlation>{CORR_THRESHOLD}: dropped {len(drop_corr)}  ->  {df_train.shape[1]} cols')

## 7. Clip bounds (IQR-3 / p99) — fit on benign train

In [ ]:
df_tr_b = df_train.loc[benign_idx]
clip_bounds, n_iqr, n_p99 = {}, 0, 0
for col in df_train.columns:
    s = df_tr_b[col]
    if s.nunique(dropna=False) <= 1:
        clip_bounds[col] = (None, float(abs(s.iloc[0])) if len(s) else None, 'const_cap')
        n_p99 += 1; continue
    q1, q3 = float(s.quantile(0.25)), float(s.quantile(0.75))
    iqr = q3 - q1
    if iqr > 0:
        clip_bounds[col] = (q1 - 3*iqr, q3 + 3*iqr, 'iqr'); n_iqr += 1
    else:
        p99 = float(s.quantile(0.99))
        clip_bounds[col] = (None, p99 if p99 > 0 else None, 'p99'); n_p99 += 1
for col, (lo, hi, st) in clip_bounds.items():
    if st == 'const_cap' and hi is not None:
        df_train[col] = df_train[col].clip(upper=hi); df_hold[col] = df_hold[col].clip(upper=hi)
    elif st in ('iqr', 'p99'):
        df_train[col] = df_train[col].clip(lower=lo, upper=hi)
        df_hold [col] = df_hold [col].clip(lower=lo, upper=hi)
print(f'IQR-clipped {n_iqr}, p99/const-clipped {n_p99}')

## 8. Flow-level StandardScaler (benign train) + clip ±10
RobustScaler was discarded — on benign-only flows the IQR can be ≈ 0 → values ~1e5 (v3 run 2 evidence).

In [ ]:
feat_cols_final = df_train.columns.tolist()
X_tr = df_train.to_numpy(dtype=np.float32, copy=True)
X_ho = df_hold .to_numpy(dtype=np.float32, copy=True)
del df_train, df_hold; gc.collect()

scaler_flow = StandardScaler().fit(X_tr[benign_idx])
X_tr = np.clip(scaler_flow.transform(X_tr), -POST_SCALE_CLIP, POST_SCALE_CLIP).astype(np.float32)
X_ho = np.clip(scaler_flow.transform(X_ho), -POST_SCALE_CLIP, POST_SCALE_CLIP).astype(np.float32)
_sc_max = max(np.abs(X_tr).max(), np.abs(X_ho).max())
print(f'flow scaler fit on {int(benign_idx.sum())} benign flows  -- max|X|={_sc_max:.3f}')

df_tr_proc = pd.DataFrame(X_tr, columns=feat_cols_final, index=meta_train.index)
df_tr_proc[['Src IP', 'session_id', 'Label', 'ts']] = meta_train[['Src IP','session_id','Label','ts']].values
df_ho_proc = pd.DataFrame(X_ho, columns=feat_cols_final, index=meta_holdout.index)
df_ho_proc[['Src IP', 'session_id', 'Label', 'ts']] = meta_holdout[['Src IP','session_id','Label','ts']].values
del X_tr, X_ho; gc.collect()

## 9. 15-second bucketing — mean + max + std + flow_count + label_multiclass

In [ ]:
def bucket_flows(df_proc, feature_cols, bucket_freq, benign_label, flow_count_max=None):
    df_proc = df_proc.copy()
    df_proc['ts']     = pd.to_datetime(df_proc['ts'])
    df_proc['bucket'] = df_proc['ts'].dt.floor(bucket_freq)
    g = ['Src IP', 'session_id', 'bucket']
    feat_agg = df_proc.groupby(g, observed=True)[feature_cols].agg(['mean','max','std'])
    feat_agg.columns = [f'{c}_{s}' for c, s in feat_agg.columns]
    feat_agg = feat_agg.reset_index()
    std_cols = [c for c in feat_agg.columns if c.endswith('_std')]
    feat_agg[std_cols] = feat_agg[std_cols].fillna(0.0)
    cnt = df_proc.groupby(g, observed=True).size().reset_index(name='_flow_count_raw')
    # binary label: 0=benign, 1=attack
    lbl = df_proc.groupby(g, observed=True)['Label'].apply(
        lambda x: 0 if (x == benign_label).all() else 1).reset_index(name='label')
    # dominant original label per bucket (0-11)
    lbl_mc = df_proc.groupby(g, observed=True)['Label'].apply(
        lambda x: int(x.mode().iloc[0])).reset_index(name='label_multiclass')
    out = feat_agg.merge(cnt, on=g).merge(lbl, on=g).merge(lbl_mc, on=g)
    if flow_count_max is None:
        flow_count_max = float(out['_flow_count_raw'].quantile(0.99))
    out['flow_count'] = (out['_flow_count_raw'] / flow_count_max).clip(upper=1.0).astype(np.float32)
    out = out.drop(columns=['_flow_count_raw'])
    # exclude both label columns from feature list
    bucket_feature_cols = [c for c in out.columns if c not in g + ['label', 'label_multiclass']]
    return out, flow_count_max, bucket_feature_cols

train_b, flow_count_max, bucket_feature_cols = bucket_flows(
    df_tr_proc, feat_cols_final, BUCKET_FREQ, BENIGN_LABEL, flow_count_max=None)
hold_b, _, _ = bucket_flows(
    df_ho_proc, feat_cols_final, BUCKET_FREQ, BENIGN_LABEL, flow_count_max=flow_count_max)
del df_tr_proc, df_ho_proc; gc.collect()

_tb_atk = int(train_b['label'].sum())
_tb_pct = train_b['label'].mean()*100
_hb_atk = int(hold_b['label'].sum())
_hb_pct = hold_b['label'].mean()*100
print(f'Train   buckets: {len(train_b):,}   attack: {_tb_atk:,} ({_tb_pct:.1f}%)')
print(f'Holdout buckets: {len(hold_b):,}    attack: {_hb_atk:,} ({_hb_pct:.1f}%)')
print(f'Features/bucket: {len(bucket_feature_cols)}  ({len(feat_cols_final)} flow cols x 3 + flow_count)')
_mc_uniq = sorted(train_b['label_multiclass'].unique().tolist())
print(f'label_multiclass unique labels (train): {_mc_uniq}')

## 10. Short-gap fill + bucket-level p1/p99 clip + bucket scaler

In [ ]:
bucket_fill = (train_b.loc[train_b['label']==0, bucket_feature_cols]
               .median().fillna(0.0).astype(np.float32))

def short_gap_fill(bdf, freq, feat_cols, fill_vals, max_gap=MAX_GAP_BUCKETS):
    f = pd.Timedelta(freq); span = max_gap * f
    def per_session(g):
        g = g.sort_values('bucket').drop_duplicates(subset=['bucket'])
        src, sid = g['Src IP'].iloc[0], g['session_id'].iloc[0]
        rows, prev = [], None
        for _, r in g.iterrows():
            if prev is not None:
                gap = r['bucket'] - prev
                if f < gap <= span + f:
                    for tb in pd.date_range(prev + f, r['bucket'] - f, freq=freq):
                        rec = {c: float(fill_vals[c]) for c in feat_cols}
                        rec.update({'Src IP': src, 'session_id': sid, 'bucket': tb,
                                    'label': 0, 'label_multiclass': 0})
                        rows.append(rec)
            rows.append(r.to_dict()); prev = r['bucket']
        return pd.DataFrame(rows)
    return (bdf.groupby(['Src IP','session_id'], group_keys=False)
               .apply(per_session).reset_index(drop=True))

train_b = short_gap_fill(train_b, BUCKET_FREQ, bucket_feature_cols, bucket_fill)
hold_b  = short_gap_fill(hold_b , BUCKET_FREQ, bucket_feature_cols, bucket_fill)
print(f'After short-gap fill (<= {MAX_GAP_BUCKETS} buckets) -- '
      f'train: {len(train_b):,}  hold: {len(hold_b):,}')

# Bucket-level clip from benign train p1/p99
_b = train_b.loc[train_b['label']==0, bucket_feature_cols]
lo = _b.quantile(0.01).astype(np.float32); hi = _b.quantile(0.99).astype(np.float32)
for c in bucket_feature_cols:
    L, H = float(lo[c]), float(hi[c])
    if not (np.isfinite(L) and np.isfinite(H) and L < H): continue
    train_b[c] = train_b[c].clip(L, H); hold_b[c] = hold_b[c].clip(L, H)
del _b; gc.collect()

# Bucket-level StandardScaler on benign train buckets
scaler_bucket = StandardScaler().fit(
    train_b.loc[train_b['label']==0, bucket_feature_cols].to_numpy(dtype=np.float32))
for bdf in (train_b, hold_b):
    arr = scaler_bucket.transform(bdf[bucket_feature_cols].to_numpy(dtype=np.float32))
    bdf[bucket_feature_cols] = np.clip(arr, -POST_SCALE_CLIP, POST_SCALE_CLIP).astype(np.float32)
_bsc_max = float(np.abs(train_b[bucket_feature_cols].values).max())
print(f'Bucket scaler fit on benign train buckets  -- max|X|={_bsc_max:.3f}')

## 11. Sliding windows (T=10, stride=2) — multiclass + window_end_ts

Returns binary label, container ID, dominant multiclass label, and **unix end-timestamp**
of each window (last bucket in the window). Timestamps enable `ordering_mode=timestamp`
in the drift-aware streaming replay.


In [ ]:
def make_windows(bucket_df, W, S, feat_cols, thr=ATTACK_FRAC_THRESHOLD):
    """Sliding windows with multiclass labels + per-window end timestamp."""
    Xs, Ys, Cs, Ys_mc, Ts = [], [], [], [], []
    for (src, sid), grp in bucket_df.groupby(['Src IP', 'session_id'], observed=True):
        grp = grp.sort_values('bucket')
        arr = grp[feat_cols].to_numpy(dtype=np.float32)
        lbl = grp['label'].to_numpy()
        lbl_mc = grp['label_multiclass'].to_numpy()
        buckets = pd.to_datetime(grp['bucket']).to_numpy()
        for k in range(0, len(arr) - W + 1, S):
            seg = lbl[k:k + W]
            seg_mc = lbl_mc[k:k + W]
            Xs.append(arr[k:k + W])
            Ys.append(int((seg != 0).mean() >= thr))
            Cs.append(src)
            Ts.append(buckets[k + W - 1])  # window_end_ts
            if (seg == 0).all():
                Ys_mc.append(0)
            else:
                atk = seg_mc[seg_mc != 0]
                vals, cnts = np.unique(atk, return_counts=True)
                Ys_mc.append(int(vals[cnts.argmax()]))
    if not Xs:
        empty_x = np.empty((0, W, len(feat_cols)), np.float32)
        empty_i8 = np.empty(0, np.int8)
        empty_obj = np.empty(0, object)
        empty_ts = np.empty(0, np.int64)
        return empty_x, empty_i8, empty_obj, empty_i8, empty_ts
    ts_unix = (pd.to_datetime(np.asarray(Ts)).astype('int64') // 10**9).astype(np.int64)
    return (np.asarray(Xs, np.float32), np.asarray(Ys, np.int8),
            np.asarray(Cs, object), np.asarray(Ys_mc, np.int8), ts_unix)

X_train_all, y_train_all, c_train_all, ymc_train_all, ts_train_all = make_windows(
    train_b, WINDOW_SIZE, STRIDE, bucket_feature_cols)
X_hold, y_hold, c_hold, ymc_hold, ts_hold = make_windows(
    hold_b, WINDOW_SIZE, STRIDE, bucket_feature_cols)
del train_b, hold_b; gc.collect()

benign_w = y_train_all == BENIGN_LABEL
X_train = X_train_all[benign_w]
y_train = y_train_all[benign_w]
c_train = c_train_all[benign_w]
del X_train_all, y_train_all, c_train_all, ymc_train_all, ts_train_all; gc.collect()

print(f'Train benign windows : {X_train.shape}')
print(f'Holdout windows      : {X_hold.shape}  attack rate {y_hold.mean()*100:.1f}%')
_hold_mc_uniq = sorted(np.unique(ymc_hold).tolist())
print(f'Holdout multiclass labels: {_hold_mc_uniq}')
print(f'Holdout ts unix range    : [{int(ts_hold.min())}, {int(ts_hold.max())}]  '
      f'span={(ts_hold.max()-ts_hold.min())/3600:.1f}h')


## 12. Stratified val / test split by (container × label)

Keeps `ts_val` / `ts_test` aligned with the same stratified split used for features and labels.


In [ ]:
from sklearn.model_selection import train_test_split
strat_key = pd.Series(c_hold.astype(str) + '|' + y_hold.astype(str))
vc = strat_key.value_counts()
_safe_key = strat_key.where(strat_key.map(vc).ge(2), other='_other')
X_val, X_test, y_val, y_test, c_val, c_test, ymc_val, ymc_test, ts_val, ts_test = train_test_split(
    X_hold, y_hold, c_hold, ymc_hold, ts_hold,
    test_size=0.5, stratify=_safe_key, random_state=RANDOM_STATE)
del X_hold, y_hold, c_hold, ymc_hold, ts_hold; gc.collect()

print(f'X_val  : {X_val.shape}   attack {y_val.mean()*100:.1f}%   containers={len(set(c_val.tolist()))}')
print(f'X_test : {X_test.shape}  attack {y_test.mean()*100:.1f}%   containers={len(set(c_test.tolist()))}')
_vmc_uniq = sorted(np.unique(ymc_val).tolist())
_tmc_uniq = sorted(np.unique(ymc_test).tolist())
print(f'y_val  multiclass labels: {_vmc_uniq}')
print(f'y_test multiclass labels: {_tmc_uniq}')
print(f'ts_test: unix [{int(ts_test.min())}, {int(ts_test.max())}]  '
      f'span={(ts_test.max()-ts_test.min())/3600:.1f}h')


## 13. Pre-save validation gates

In [ ]:
_xmax  = float(max(np.abs(X_train).max(), np.abs(X_val).max(), np.abs(X_test).max()))
_xmean = float(np.abs(X_train).mean())
_nan   = bool(np.isnan(X_train).any() or np.isnan(X_val).any() or np.isnan(X_test).any())

print('Window scale check')
print(f'  mean|X_train|={_xmean:.4f}  max|X|={_xmax:.4f}  nan={_nan}')
if _nan or _xmax > 50 or _xmean > 20:
    raise ValueError(f'BAD SCALE: re-run from \u00a78 StandardScaler + clip; max|X|={_xmax}')
if (y_train != 0).any():
    raise ValueError('Train windows must be benign-only')

overlap = train_sessions & holdout_sessions
assert len(overlap) == 0, 'Session leakage between train and holdout!'
print('Session leakage      : PASS')
print(f'Train benign-only    : PASS ({(y_train==0).all()})')
print(f'Holdout attack rate  : {y_val.mean()*100:.1f}% val  /  {y_test.mean()*100:.1f}% test')

# MC sanity: every attack window must have a non-zero multiclass label
_mc_bad = int(((y_test == 1) & (ymc_test == 0)).sum())
assert _mc_bad == 0, f'{_mc_bad} attack windows have label_multiclass=0'
print(f'MC label sanity      : PASS (no attack window with multiclass=0)')

# Timestamp integrity (Task 6) — required for temporal streaming replay
assert len(ts_val) == len(y_val) and len(ts_test) == len(y_test), 'ts length mismatch'
assert np.isfinite(ts_test.astype(np.float64)).all(), 'ts_test has non-finite values'
assert np.nanstd(ts_test.astype(np.float64)) > 0, 'ts_test has zero variance'
print(f'Timestamp integrity  : PASS  ts_test span={(ts_test.max()-ts_test.min())/3600:.1f}h')


## 14. Drift baseline (unchanged from vNext — for reference)

Per-feature mean / std / quantiles of benign train — used by model to compute PSI and KS.

In [ ]:
F = X_train.shape[2]
flat_train = X_train.reshape(-1, F)
drift_quantiles = np.linspace(0.0, 1.0, 11)
drift_baseline = {
    'mean'      : flat_train.mean(axis=0).astype(np.float32),
    'std'       : flat_train.std(axis=0).astype(np.float32) + 1e-6,
    'q'         : np.quantile(flat_train, drift_quantiles, axis=0).astype(np.float32),
    'q_levels'  : drift_quantiles.astype(np.float32),
    'feat_names': bucket_feature_cols,
}
print(f'drift baseline: features={F}  quantile levels={len(drift_quantiles)}')
print(f'  mean range  [{drift_baseline["mean"].min():.3f}, {drift_baseline["mean"].max():.3f}]')
print(f'  std  range  [{drift_baseline["std"].min():.3f},  {drift_baseline["std"].max():.3f}]')
print('NOTE: drift baseline is identical to vNext run — only saved here for completeness.')

## 15. Save `windows_vnext_mc.npz` (multiclass + timestamps)

Does **not** overwrite preproc pkl, drift baseline, or manifest from the binary vNext run.


In [ ]:
import joblib

# Save only the mc npz — do NOT overwrite preproc pkl, drift baseline, or manifest
npz_mc_path = f'{OUTPUT_DIR}/windows_vnext_mc.npz'
np.savez_compressed(
    npz_mc_path,
    X_train=X_train,
    X_val=X_val,
    X_test=X_test,
    y_val=y_val,
    y_test=y_test,
    y_val_multiclass=ymc_val.astype(np.int8),
    y_test_multiclass=ymc_test.astype(np.int8),
    c_val=c_val.astype(str),
    c_test=c_test.astype(str),
    ts_val=ts_val.astype(np.int64),
    ts_test=ts_test.astype(np.int64),
)
_sz = os.path.getsize(npz_mc_path) / 1024 / 1024
print(f'Saved windows_vnext_mc.npz -> {npz_mc_path}  ({_sz:.1f} MB)')
print('Keys: X_train, X_val, X_test, y_val, y_test,')
print('      y_val_multiclass, y_test_multiclass, c_val, c_test, ts_val, ts_test')
print(f'ts_test unix range: [{int(ts_test.min())}, {int(ts_test.max())}]')
print()
print('Drift baseline, preproc pkl, manifest are UNCHANGED from the original vNext run.')
print('Only windows_vnext_mc.npz is new/updated.')
print('NOTE: For drift-aware timestamp ordering, also re-run mdc_preprocess_vNext.ipynb')
print('      so windows_vnext.npz contains ts_val/ts_test (drift notebook loads that file).')


# Task 10 — export official label map (embedded; no separate .py)
_LABEL_MAP = {
    '0': 'BENIGN', '1': 'CVE-2020-13379', '2': 'Node-RED Recon',
    '3': 'Node-RED RCE', '4': 'Node-RED Escape', '5': 'CVE-2021-43798',
    '6': 'CVE-2019-20933', '7': 'CVE-2021-30465', '8': 'CVE-2021-25741',
    '9': 'CVE-2022-23648', '10': 'CVE-2019-5736', '11': 'DSB Nuclei Scan',
}
_map_path = f'{OUTPUT_DIR}/mdc_label_map.json'
with open(_map_path, 'w', encoding='utf-8') as fh:
    json.dump({
        'label_names': _LABEL_MAP,
        'citation': 'Sever & Dogan (2023), ITU Journal / MDC Kaggle dataset',
        'source': 'embedded in mdc_preprocess_vNext_mc.ipynb',
    }, fh, indent=2)
print(f'Saved label map -> {_map_path}')


## 16. Push to Drive `vNEXT_test/processed/` (FUSE-safe, verified)

Copies `windows_vnext_mc.npz` + `mdc_label_map.json` to:
**My Drive → vNEXT_test → processed**

Also writes `vNEXT_test/SYNC_RECEIPT.json` so the Drive UI shows a sync marker.


In [ ]:
# Drive I/O helpers — readable source (best practice for notebooks)
# Run this cell once before any Drive pull/push.
# Source of truth in repo: notebook/mdc_drive_io.py (local tests only).

"""
MDC Colab Drive I/O — FUSE-safe push/fetch for project folder ``vNEXT_test``.

Why this exists (past failures):
1. ``force_remount=True`` → "Mountpoint must not already contain files"
2. Writing only to ``/content/drive/MyDrive/...`` → notebook says OK, Drive UI empty
3. Stale ``/content/drive`` without real FUSE → files "found" that are not on Drive
4. Scattered paths (Module4_MDC/runs/vnext_run vs shared folder IDs)

Rules:
- ALWAYS stage artifacts under ``/content/vNEXT_test_local/`` first
- THEN copy to ``MyDrive/vNEXT_test/{processed|runs|drift_aware}/``
- VERIFY destination size matches source
- Write ``SYNC_RECEIPT.json`` so the UI has a small visible marker file
- NEVER call ``force_remount=True`` by default
"""
from __future__ import annotations

import json
import os
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple, Union

PROJECT_FOLDER = "vNEXT_test"
LOCAL_ROOT_NAME = "vNEXT_test_local"

SUBDIR_PROCESSED = "processed"
SUBDIR_RUNS = "runs"
SUBDIR_DRIFT = "drift_aware"

PathLike = Union[str, Path]


def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def mount_drive_safe(*, force_remount: bool = False) -> Path:
    """
    Gentle Drive mount (v3 pattern).

    - Mount only if ``MyDrive`` is missing
    - Never use force_remount unless explicitly requested
    - Verify mount by listing MyDrive (not just isdir)
    """
    if not in_colab():
        raise RuntimeError("mount_drive_safe() is for Colab only")

    from google.colab import drive

    mount = "/content/drive"
    mydrive = Path(mount) / "MyDrive"

    if force_remount:
        # Last resort only — can fail if mountpoint already has files
        drive.mount(mount, force_remount=True)
    elif not mydrive.is_dir():
        os.makedirs(mount, exist_ok=True)
        drive.mount(mount)
    else:
        print("Drive already mounted — skipping drive.mount() (no force_remount)")

    if not mydrive.is_dir():
        raise RuntimeError(
            "Drive mount failed: /content/drive/MyDrive missing. "
            "Runtime → Restart runtime, then re-run and complete auth."
        )

    # Prove FUSE is live (empty placeholder dirs can fool isdir)
    try:
        _ = list(mydrive.iterdir())
    except OSError as e:
        raise RuntimeError(
            f"Drive MyDrive exists but is not readable ({e}). "
            "Restart runtime and remount."
        ) from e

    print(f"Drive OK: {mydrive}")
    return mydrive


def project_paths(mydrive: Optional[Path] = None) -> Dict[str, Path]:
    """
    Canonical layout::

        MyDrive/vNEXT_test/
          processed/
          runs/
          drift_aware/

        /content/vNEXT_test_local/
          processed/
          runs/
          drift_aware/
    """
    if in_colab():
        if mydrive is None:
            mydrive = Path("/content/drive/MyDrive")
        drive_root = mydrive / PROJECT_FOLDER
        local_root = Path("/content") / LOCAL_ROOT_NAME
    else:
        # Local/dev fallback under repo outputs
        drive_root = Path("../outputs") / PROJECT_FOLDER
        local_root = Path("../outputs") / f"{LOCAL_ROOT_NAME}"

    paths = {
        "drive_root": drive_root,
        "local_root": local_root,
        "drive_processed": drive_root / SUBDIR_PROCESSED,
        "drive_runs": drive_root / SUBDIR_RUNS,
        "drive_drift": drive_root / SUBDIR_DRIFT,
        "local_processed": local_root / SUBDIR_PROCESSED,
        "local_runs": local_root / SUBDIR_RUNS,
        "local_drift": local_root / SUBDIR_DRIFT,
    }
    return paths


def ensure_project_dirs(paths: Optional[Dict[str, Path]] = None) -> Dict[str, Path]:
    paths = paths or project_paths()
    for key, p in paths.items():
        if key.endswith("_root") or key.startswith("drive_") or key.startswith("local_"):
            p.mkdir(parents=True, exist_ok=True)
    return paths


def ui_path(*parts: str) -> str:
    """Human path for Drive web UI."""
    bits = ["My Drive", PROJECT_FOLDER, *[str(p) for p in parts if p]]
    return " > ".join(bits)


def _fsync_file(path: Path) -> None:
    try:
        with open(path, "rb") as fh:
            os.fsync(fh.fileno())
    except OSError:
        pass


def push_file(
    src: PathLike,
    dest: PathLike,
    *,
    overwrite: bool = True,
) -> Dict[str, Any]:
    """
    Copy one file to Drive (or any dest) and verify byte size.

    Returns a result dict; raises on hard failure.
    """
    src = Path(src)
    dest = Path(dest)
    if not src.is_file():
        raise FileNotFoundError(f"push_file: source missing: {src}")

    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and not overwrite:
        return {
            "ok": True,
            "skipped": True,
            "src": str(src),
            "dest": str(dest),
            "src_size": src.stat().st_size,
            "dest_size": dest.stat().st_size,
        }

    # Copy to a temp name then replace — reduces half-written FUSE files
    tmp = dest.with_suffix(dest.suffix + ".partial")
    shutil.copy2(src, tmp)
    _fsync_file(tmp)
    os.replace(tmp, dest)
    _fsync_file(dest)

    src_sz = src.stat().st_size
    # Small settle for FUSE metadata
    time.sleep(0.05)
    if not dest.is_file():
        raise RuntimeError(f"push_file: dest missing after copy: {dest}")
    dest_sz = dest.stat().st_size
    ok = src_sz == dest_sz
    result = {
        "ok": ok,
        "skipped": False,
        "src": str(src),
        "dest": str(dest),
        "src_size": src_sz,
        "dest_size": dest_sz,
        "name": dest.name,
    }
    if not ok:
        raise RuntimeError(
            f"SIZE MISMATCH after push: {dest.name} "
            f"src={src_sz:,} dest={dest_sz:,}. "
            "FUSE sync failed — retry push or restart runtime."
        )
    return result


def push_files(
    pairs: Sequence[Tuple[PathLike, PathLike]],
) -> List[Dict[str, Any]]:
    results = []
    for src, dest in pairs:
        results.append(push_file(src, dest))
    return results


def push_tree(src_dir: PathLike, dest_dir: PathLike) -> List[Dict[str, Any]]:
    src_dir = Path(src_dir)
    dest_dir = Path(dest_dir)
    if not src_dir.is_dir():
        raise FileNotFoundError(f"push_tree: missing {src_dir}")
    results = []
    for f in sorted(src_dir.rglob("*")):
        if f.is_file():
            rel = f.relative_to(src_dir)
            results.append(push_file(f, dest_dir / rel))
    return results


def pull_file(src: PathLike, dest: PathLike) -> Dict[str, Any]:
    """Copy from Drive (or any src) into local cache.

    No-op when src and dest are the same path (e.g. gdown already wrote into
    the local staging folder).
    """
    src = Path(src).resolve()
    dest = Path(dest).resolve()
    if not src.is_file():
        raise FileNotFoundError(f"pull_file: source missing: {src}")
    if src == dest:
        return {
            "ok": True,
            "skipped": True,
            "src": str(src),
            "dest": str(dest),
            "size": src.stat().st_size,
        }
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dest)
    if dest.stat().st_size != src.stat().st_size:
        raise RuntimeError(f"pull_file size mismatch: {src} -> {dest}")
    return {"ok": True, "skipped": False, "src": str(src), "dest": str(dest), "size": dest.stat().st_size}


def first_existing(name: str, roots: Sequence[PathLike]) -> Optional[Path]:
    for root in roots:
        if root is None:
            continue
        p = Path(root) / name
        try:
            if p.is_file():
                return p
        except OSError:
            continue
    return None


def pull_required(
    names: Sequence[str],
    search_roots: Sequence[PathLike],
    local_dir: PathLike,
) -> Dict[str, Path]:
    """
    Resolve required artifacts into ``local_dir`` (FUSE-safe load path).
    """
    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)
    out: Dict[str, Path] = {}
    missing = []
    for name in names:
        # Prefer already-local copy if size>0
        local_hit = local_dir / name
        if local_hit.is_file() and local_hit.stat().st_size > 0:
            out[name] = local_hit
            print(f"  OK local  {name} ({local_hit.stat().st_size:,} bytes)")
            continue
        src = first_existing(name, search_roots)
        if src is None:
            missing.append(name)
            continue
        pull_file(src, local_hit)
        out[name] = local_hit
        print(f"  OK pulled {name} <- {src} ({local_hit.stat().st_size:,} bytes)")
    if missing:
        raise FileNotFoundError(
            "Missing required artifacts:\n  - "
            + "\n  - ".join(missing)
            + f"\nSearched: {[str(r) for r in search_roots]}\n"
            f"Expected under Drive UI: {ui_path(SUBDIR_PROCESSED)} or {ui_path(SUBDIR_RUNS)}"
        )
    return out


def write_sync_receipt(
    drive_root: PathLike,
    *,
    notebook: str,
    results: Sequence[Dict[str, Any]],
    extra: Optional[Dict[str, Any]] = None,
) -> Path:
    """
    Small JSON marker on Drive so the web UI always shows *something*
    after a push (helps debug 'I pushed but UI is empty').
    """
    drive_root = Path(drive_root)
    drive_root.mkdir(parents=True, exist_ok=True)
    receipt = {
        "project": PROJECT_FOLDER,
        "notebook": notebook,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "files": [
            {
                "name": r.get("name") or Path(r.get("dest", "")).name,
                "dest": r.get("dest"),
                "size": r.get("dest_size"),
                "ok": r.get("ok"),
            }
            for r in results
        ],
        "ui_hint": f"Open Google Drive → {ui_path()}",
        "extra": extra or {},
    }
    path = drive_root / "SYNC_RECEIPT.json"
    # Write via local temp then push for FUSE safety
    tmp_local = Path("/tmp") / f"SYNC_RECEIPT_{int(time.time())}.json"
    if not in_colab():
        tmp_local = Path(".") / f"SYNC_RECEIPT_{int(time.time())}.json"
    tmp_local.write_text(json.dumps(receipt, indent=2), encoding="utf-8")
    push_file(tmp_local, path)
    try:
        tmp_local.unlink()
    except OSError:
        pass
    return path


def list_dir_safe(path: PathLike) -> List[str]:
    path = Path(path)
    if not path.is_dir():
        return []
    try:
        return sorted(p.name for p in path.iterdir())
    except OSError:
        return []


def print_push_report(
    results: Sequence[Dict[str, Any]],
    *,
    ui_folder: str,
) -> None:
    print("\n=== Drive push report ===")
    ok_n = sum(1 for r in results if r.get("ok"))
    print(f"  pushed_ok: {ok_n}/{len(results)}")
    for r in results:
        flag = "OK" if r.get("ok") else "FAIL"
        print(
            f"  [{flag}] {r.get('name', '?')}: "
            f"{r.get('dest_size', 0):,} bytes -> {r.get('dest')}"
        )
    print(f"\n  Drive UI path: {ui_folder}")
    print("  If UI empty: wait 1–2 min, hard-refresh Drive, confirm SAME Google account.")
    print("  Look for SYNC_RECEIPT.json in the project root as a sync marker.")


def bootstrap_paths_colab(notebook: str = "unknown") -> Dict[str, Path]:
    """One-call setup used by notebooks after imports."""
    mydrive = mount_drive_safe()
    paths = ensure_project_dirs(project_paths(mydrive))
    print(f"[{notebook}] project folder: {paths['drive_root']}")
    print(f"[{notebook}] local staging : {paths['local_root']}")
    print(f"[{notebook}] UI           : {ui_path()}")
    return paths

# ---------------------------------------------------------------------------
# Notebook binding — call sites use `dio.*` (same API as the repo module)
# ---------------------------------------------------------------------------
import types as _types

dio = _types.SimpleNamespace(
    PROJECT_FOLDER=PROJECT_FOLDER,
    LOCAL_ROOT_NAME=LOCAL_ROOT_NAME,
    SUBDIR_PROCESSED=SUBDIR_PROCESSED,
    SUBDIR_RUNS=SUBDIR_RUNS,
    SUBDIR_DRIFT=SUBDIR_DRIFT,
    in_colab=in_colab,
    mount_drive_safe=mount_drive_safe,
    project_paths=project_paths,
    ensure_project_dirs=ensure_project_dirs,
    ui_path=ui_path,
    push_file=push_file,
    push_files=push_files,
    push_tree=push_tree,
    pull_file=pull_file,
    first_existing=first_existing,
    pull_required=pull_required,
    write_sync_receipt=write_sync_receipt,
    list_dir_safe=list_dir_safe,
    print_push_report=print_push_report,
    bootstrap_paths_colab=bootstrap_paths_colab,
)
print("Drive I/O helpers ready (readable cell — not a base64 blob)")

npz_mc_path = Path(OUTPUT_DIR) / 'windows_vnext_mc.npz'
map_path = Path(OUTPUT_DIR) / 'mdc_label_map.json'

if not npz_mc_path.is_file():
    raise FileNotFoundError(f'Missing {npz_mc_path} — run save cell first')

if _IN_COLAB:
    if 'dio' not in globals():
        raise RuntimeError('Run the Drive I/O helpers cell first')
    paths = dio.bootstrap_paths_colab(notebook='mdc_preprocess_vNext_mc')
    pairs = [(npz_mc_path, paths['drive_processed'] / npz_mc_path.name)]
    if map_path.is_file():
        pairs.append((map_path, paths['drive_processed'] / map_path.name))
    results = dio.push_files(pairs)
    receipt = dio.write_sync_receipt(
        paths['drive_root'], notebook='mdc_preprocess_vNext_mc', results=results,
        extra={'pushed_subdir': 'processed'},
    )
    dio.print_push_report(results, ui_folder=dio.ui_path('processed'))
    print(f'SYNC_RECEIPT -> {receipt}')
    print('Verify: My Drive > vNEXT_test > processed > windows_vnext_mc.npz')
else:
    dest = Path(f'../outputs/{DRIVE_PROJECT_FOLDER}/processed')
    dest.mkdir(parents=True, exist_ok=True)
    import shutil
    shutil.copy2(npz_mc_path, dest / npz_mc_path.name)
    if map_path.is_file():
        shutil.copy2(map_path, dest / map_path.name)
    print(f'Local copy -> {dest}')
